<a href="https://colab.research.google.com/github/Antohardo/VHE_Python/blob/main/%D0%A0%D1%83%D0%B1%D1%86%D0%BE%D0%B2_%D0%90%D0%BD%D1%82%D0%BE%D0%BD_%D0%92%D0%A8%D0%AD_%D0%94%D0%97_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from datetime import datetime
import math
from typing import Any, Optional
import pandas as pd

In [ ]:
class Account:
    """Базовый класс банковского счета."""

    def __init__(self, account_holder: str, balance: float = 0.0) -> None:
        """Создает счет с именем владельца и начальным балансом."""
        # проверяем что имя владельца не пустое
        if not isinstance(account_holder, str) or not account_holder.strip():
            raise ValueError("Имя владельца счета не должно быть пустой строкой!")

        self.holder: str = account_holder.strip()
        self.operations_history: list[dict[str, Any]] = []

        # проверка баланса вынесена в отдельный метод для переопределения в CreditAccount
        self._validate_initial_balance(balance)
        # прибавляем 0.0, чтобы избавиться от возможного -0.0 после округления
        self._balance: float = round(float(balance), 2) + 0.0

    @staticmethod
    def _is_finite_number(value: Any) -> bool:
        """Проверяет, что передано конечное число (не bool, не NaN и не inf)."""
        return (
            isinstance(value, (int, float))
            and not isinstance(value, bool)
            and math.isfinite(value)
        )

    def _validate_initial_balance(self, balance: float) -> None:
        """Проверяет корректность начального баланса."""
        # проверяем balance < 0 до округления, чтобы не пропускать значения вроде -0.001
        if not self._is_finite_number(balance) or balance < 0:
            raise ValueError("Начальный баланс не может быть отрицательным.")

    def _record_operation(
        self,
        op_type: str,
        amount: float,
        status: str,
        **extra_data: Any,
    ) -> dict[str, Any]:
        """Формирует запись об операции и сохраняет ее в историю."""
        record: dict[str, Any] = {
            "type": op_type,
            "amount": round(float(amount), 2) + 0.0,
            "datetime": datetime.now(),
            "balance_after": self._balance,
            "status": status,
        }
        record.update(extra_data)
        self.operations_history.append(record)
        return record

    def deposit(self, amount: float) -> bool:
        """Пополняет счет на указанную сумму."""
        if not self._is_finite_number(amount):
            self._record_operation("deposit", 0.0, "fail")
            return False

        # сначала округляем до копеек, чтобы не проходили суммы вроде 0.001
        clean_amount = round(float(amount), 2) + 0.0
        if clean_amount <= 0:
            self._record_operation("deposit", clean_amount, "fail")
            return False

        self._balance = round(self._balance + clean_amount, 2) + 0.0
        self._record_operation("deposit", clean_amount, "success")
        return True

    def withdraw(self, amount: float) -> bool:
        """Снимает средства со счета при наличии достаточного баланса."""
        if not self._is_finite_number(amount):
            self._record_operation("withdraw", 0.0, "fail")
            return False

        clean_amount = round(float(amount), 2) + 0.0
        if clean_amount <= 0:
            self._record_operation("withdraw", clean_amount, "fail")
            return False

        # если на счете недостаточно средств, пишем fail в историю
        if clean_amount > self._balance:
            self._record_operation("withdraw", clean_amount, "fail")
            return False

        self._balance = round(self._balance - clean_amount, 2) + 0.0
        self._record_operation("withdraw", clean_amount, "success")
        return True

    def get_balance(self) -> float:
        """Возвращает текущий баланс счета."""
        return self._balance

    def get_history(
        self,
        op_type: Optional[str] = None,
        status: Optional[str] = None,
    ) -> list[dict[str, Any]]:
        """Возвращает историю операций с возможностью фильтрации по типу и статусу."""
        result: list[dict[str, Any]] = []
        for op in self.operations_history:
            if op_type is not None and op["type"] != op_type:
                continue
            if status is not None and op["status"] != status:
                continue
            op_copy = op.copy()
            # добавляем дату в виде строки для удобного чтения
            op_copy["datetime_str"] = op["datetime"].strftime("%Y-%m-%d %H:%M:%S")
            result.append(op_copy)
        return result

    def get_analytics(self) -> dict[str, Any]:
        """Возвращает сводную аналитику по истории операций счета."""
        total_deposited = sum(
            op["amount"] for op in self.operations_history
            if op["type"] == "deposit" and op["status"] == "success"
        )
        total_withdraw = sum(
            op["amount"] for op in self.operations_history
            if op["type"] == "withdraw" and op["status"] == "success"
        )
        success_count = sum(1 for op in self.operations_history if op["status"] == "success")
        fail_count = sum(1 for op in self.operations_history if op["status"] == "fail")

        return {
            "holder": self.holder,
            "current_balance": self._balance,
            "total_operations": len(self.operations_history),
            "successful_operations": success_count,
            "failed_operations": fail_count,
            "total_deposited": round(total_deposited, 2) + 0.0,
            "total_withdraw": round(total_withdraw, 2) + 0.0,
            "net_cash_flow": round(total_deposited - total_withdraw, 2) + 0.0,
        }

    def transfer_to(self, target_account: "Account", amount: float) -> bool:
        """Переводит средства с текущего счета на другой счет."""
        if not isinstance(target_account, Account) or target_account is self:
            return False

        prev_balance = self._balance
        if not self.withdraw(amount):
            return False

        # если счет получателя отклонил пополнение, откатываем списание у отправителя
        if not target_account.deposit(amount):
            self._balance = prev_balance
            last_op = self.operations_history[-1]
            last_op["status"] = "fail"
            last_op["balance_after"] = self._balance
            if "credit_used" in last_op:
                last_op["credit_used"] = False
            return False

        return True

    def __str__(self) -> str:
        """Возвращает строковое представление счета."""
        return f"Счет (владелец: {self.holder}, баланс: {self._balance:.2f})"

In [ ]:
class CreditAccount(Account):
    """Кредитный счет с поддержкой кредитного лимита."""

    def __init__(
        self,
        account_holder: str,
        balance: float = 0.0,
        credit_limit: float = 0.0,
    ) -> None:
        """Создает кредитный счет с заданным лимитом."""
        if not self._is_finite_number(credit_limit) or credit_limit < 0:
            raise ValueError("Кредитный лимит не может быть отрицательным.")

        # сохраняем лимит до вызова super(), т.к. он нужен в _validate_initial_balance
        self._credit_limit: float = round(float(credit_limit), 2) + 0.0
        super().__init__(account_holder, balance)

    def _validate_initial_balance(self, balance: float) -> None:
        """Проверяет, что начальный баланс не ниже отрицательного кредитного лимита."""
        if not self._is_finite_number(balance) or balance < -self._credit_limit:
            raise ValueError(
                f"Начальный баланс не может быть ниже -{self._credit_limit}."
            )

    def get_credit_limit(self) -> float:
        """Возвращает размер кредитного лимита."""
        return self._credit_limit

    def get_available_credit(self) -> float:
        """Возвращает сумму доступных средств с учетом кредитного лимита."""
        return round(self._balance + self._credit_limit, 2) + 0.0

    def deposit(self, amount: float) -> bool:
        """Пополняет кредитный счет и фиксирует операцию в истории."""
        if not self._is_finite_number(amount):
            self._record_operation("deposit", 0.0, "fail", credit_used=False)
            return False

        clean_amount = round(float(amount), 2) + 0.0
        if clean_amount <= 0:
            self._record_operation("deposit", clean_amount, "fail", credit_used=False)
            return False

        self._balance = round(self._balance + clean_amount, 2) + 0.0
        self._record_operation("deposit", clean_amount, "success", credit_used=False)
        return True

    def withdraw(self, amount: float) -> bool:
        """Снимает средства в пределах доступного лимита и отмечает использование кредита."""
        if not self._is_finite_number(amount):
            self._record_operation("withdraw", 0.0, "fail", credit_used=False)
            return False

        clean_amount = round(float(amount), 2) + 0.0
        if clean_amount <= 0:
            self._record_operation("withdraw", clean_amount, "fail", credit_used=False)
            return False

        # проверяем, хватает ли средств с учетом кредитного лимита
        if clean_amount > self.get_available_credit():
            self._record_operation("withdraw", clean_amount, "fail", credit_used=False)
            return False

        self._balance = round(self._balance - clean_amount, 2) + 0.0
        # если баланс ушел в минус, значит использовали кредитные средства
        credit_used = self._balance < 0
        self._record_operation("withdraw", clean_amount, "success", credit_used=credit_used)
        return True

    def get_analytics(self) -> dict[str, Any]:
        """Возвращает аналитику по счету с учетом кредитного лимита и текущего долга."""
        analytics = super().get_analytics()
        credit_ops = sum(
            1 for op in self.operations_history if op.get("credit_used") is True
        )
        debt = abs(self._balance) if self._balance < 0 else 0.0

        analytics.update({
            "credit_limit": self._credit_limit,
            "available_funds": self.get_available_credit(),
            "current_credit_debt": round(debt, 2) + 0.0,
            "operations_with_credit_count": credit_ops,
        })
        return analytics

    def __str__(self) -> str:
        """Возвращает строковое представление кредитного счета."""
        return (
            f"КредитныйСчет(владелец: {self.holder}, баланс: {self._balance:.2f}, "
            f"лимит: {self._credit_limit:.2f}, доступно: {self.get_available_credit():.2f})"
        )

In [ ]:
acc = Account("Антон", 1000)
print("Создан:", acc)

acc.deposit(500)
acc.withdraw(300)
acc.withdraw(5000)
acc.deposit(-200)

print("Текущтй баланс через get_balance():", acc.get_balance())
print("\nАналитика по счету:")
for key, value in acc.get_analytics().items():
    print(f" {key}: {value}")

pd.DataFrame(acc.get_history())


Создан: Счет (владелец: Антон, баланс: 1000.00)
Текущтй баланс через get_balance(): 1200.0

Аналитика по счету:
 holder: Антон
 current_balance: 1200.0
 total_operations: 4
 successful_operations: 2
 failed_operations: 2
 total_deposited: 500.0
 total_withdraw: 300.0
 net_cash_flow: 200.0


,type,amount,datetime,balance_after,status,datetime_str
0,deposit,500.0,2026-10-28 22:07:02.835116,1500.0,success,2026-10-28 22:07:02
1,withdraw,300.0,2026-10-28 22:07:02.835172,1200.0,success,2026-10-28 22:07:02
2,withdraw,5000.0,2026-10-28 22:07:02.835211,1200.0,fail,2026-10-28 22:07:02
3,deposit,-200.0,2026-10-28 22:07:02.835254,1200.0,fail,2026-10-28 22:07:02


In [ ]:
# проверяем работу кредитного счета
credit_acc = CreditAccount("Антон (Кредитный)", balance=500, credit_limit=1000)
print("Создан:", credit_acc)

credit_acc.withdraw(300)   # хватает своих денег
credit_acc.withdraw(600)   # уходим в кредит на -400
credit_acc.withdraw(700)   # больше лимита, операция не пройдет
credit_acc.deposit(250)    # частичное погашение

# переводим 150 с первого счета, чтобы закрыть остаток долга
acc.transfer_to(credit_acc, 150)

print("После всех операций:", credit_acc)
print("Доступный кредитный остаток:", credit_acc.get_available_credit())

print("\nАналитика кредитного счета:")
for key, value in credit_acc.get_analytics().items():
    print(f"  {key}: {value}")

pd.DataFrame(credit_acc.get_history())

Создан: КредитныйСчет(владелец: Антон (Кредитный), баланс: 500.00, лимит: 1000.00, доступно: 1500.00)
После всех операций: КредитныйСчет(владелец: Антон (Кредитный), баланс: 0.00, лимит: 1000.00, доступно: 1000.00)
Доступный кредитный остаток: 1000.0

Аналитика кредитного счета:
  holder: Антон (Кредитный)
  current_balance: 0.0
  total_operations: 5
  successful_operations: 4
  failed_operations: 1
  total_deposited: 400.0
  total_withdraw: 900.0
  net_cash_flow: -500.0
  credit_limit: 1000.0
  available_funds: 1000.0
  current_credit_debt: 0.0
  operations_with_credit_count: 1


,type,amount,datetime,balance_after,status,credit_used,datetime_str
0,withdraw,300.0,2026-10-28 22:07:02.895730,200.0,success,False,2026-10-28 22:07:02
1,withdraw,600.0,2026-10-28 22:07:02.895864,-400.0,success,True,2026-10-28 22:07:02
2,withdraw,700.0,2026-10-28 22:07:02.895945,-400.0,fail,False,2026-10-28 22:07:02
3,deposit,250.0,2026-10-28 22:07:02.896034,-150.0,success,False,2026-10-28 22:07:02
4,deposit,150.0,2026-10-28 22:07:02.896129,0.0,success,False,2026-10-28 22:07:02


In [ ]:
# проверка нестандартных случаев

# 1. попытка создать счет с микро-минусом (-0.001 не должно превращаться в -0.0)
try:
    bad_acc = Account("Тест", -0.001)
except ValueError as e:
    print("Тест 1 (баланс -0.001):", e)

# 2. проверка на NaN, бесконечность, bool и суммы меньше копейки
edge_acc = Account("Тест Границ", 100)
edge_acc.deposit(float("nan"))   # NaN
edge_acc.deposit(float("inf"))   # бесконечность
edge_acc.deposit(True)
edge_acc.deposit(0.001)          # после округления дает 0.00, операция не должна проходить как успешная

print("Тест 2 (баланс после некорректных пополнений остался 100.0):", edge_acc.get_balance())

# 3. проверка отката перевода, если счет получателя заблокирован
class FrozenAccount(Account):
    """Тестовый класс замороженного счета, который отклоняет любые пополнения."""

    def deposit(self, amount: float) -> bool:
        self._record_operation("deposit", 0.0, "fail")
        return False

frozen_acc = FrozenAccount("Замороженный клиент", 50)
transfer_result = edge_acc.transfer_to(frozen_acc, 40)

print("Тест 3 (результат перевода на замороженный счет):", transfer_result)
print("Баланс отправителя не пострадал (откат сработал):", edge_acc.get_balance())

# 4. проверка защиты истории от изменений извне и фильтрации
history_copy = edge_acc.get_history(status="fail")
history_copy.clear()  # очищаем полученный список
print("Тест 4 (записей в реальной истории после очистки копии):", len(edge_acc.get_history()))

pd.DataFrame(edge_acc.get_history(status="fail"))

Тест 1 (баланс -0.001): Начальный баланс не может быть отрицательным.
Тест 2 (баланс после некорректных пополнений остался 100.0): 100.0
Тест 3 (результат перевода на замороженный счет): False
Баланс отправителя не пострадал (откат сработал): 100.0
Тест 4 (записей в реальной истории после очистки копии): 5


,type,amount,datetime,balance_after,status,datetime_str
0,deposit,0.0,2026-10-28 22:07:02.959754,100.0,fail,2026-10-28 22:07:02
1,deposit,0.0,2026-10-28 22:07:02.959881,100.0,fail,2026-10-28 22:07:02
2,deposit,0.0,2026-10-28 22:07:02.959961,100.0,fail,2026-10-28 22:07:02
3,deposit,0.0,2026-10-28 22:07:02.960032,100.0,fail,2026-10-28 22:07:02
4,withdraw,40.0,2026-10-28 22:07:02.960572,100.0,fail,2026-10-28 22:07:02
